# 1. Imports

In [1]:
import datetime as dt
import os
from time import sleep

import pandas as pd
import requests
from bs4 import BeautifulSoup
from dotenv import load_dotenv
from lat_lon_parser import parse
from pytz import timezone

load_dotenv();

# 2. Function definitions

## Define connection string for local database

In [2]:
def connect_to_gans_local() -> str:
    """
    Description: returns connection string for gans schema on local instance.
                 only runs if the connection_password has been imported from keys.py
    Parameters:
        None
    Returns:
        connection_string | str: string for SQLAlchemy connections
    """
    schema = "gans_db"
    host = "127.0.0.1"
    user = "root"
    password = os.getenv("MYSQL_PASSWORD")
    port = 3306

    return f'mysql+pymysql://{user}:{password}@{host}:{port}/{schema}'

## Scrape data for cities table

In [3]:
def scrape_and_send_city_data(
        cities_list: list[str],
        connection_string: str,
        ) -> None:
    """
    Description: scrapes latitude, logitude, and country from Wikipedia for provided cities
                 and sends them to a connected database
    Parameters:
        cities_list | list : list of city names
        connection_string | str: string for SQLAlchemy connections
    Returns:
        None, just sends data to the database
    """
    # Get connection string
    connection_string = connect_to_gans_local()

    # Create empty list for storing data
    cities_data = []

    # Loop through cities
    for city in cities_list:

        # Create url
        url = "https://en.wikipedia.org/wiki/" + city
        headers = {'User-Agent': 'Chrome/134.0.0.0'}

        # Pull HTML
        response = requests.get(url, headers=headers)
        city_soup = BeautifulSoup(response.content, 'html.parser')

        # Locate and extract data
        country = city_soup.find(class_="infobox-data").get_text()
        lat = parse(city_soup.find('span', class_='latitude').get_text())
        long = parse(city_soup.find('span', class_='longitude').get_text())

        # Create dictionary for 1 row of final data frame
        city_data = {
            'city': city,
            'country': country,
            'latitude': lat,
            'longitude': long,
        }
        # Add row to list
        cities_data.append(city_data)

    # Combine all rows into data frame
    cities_df = pd.DataFrame(cities_data)

    # Send data to database
    cities_df.to_sql('cities',
                     if_exists='append',
                     con=connection_string,
                     index=False)

## Scrape data for cities populations table

In [4]:
def scrape_and_send_population_data(connection_string: str) -> None:
    """
    Description: scrapes population from Wikipedia for cities in the database,
                 then sends data to a connected database
    Parameters:
        connection_string | str: string for SQLAlchemy connections
    Returns:
        None, just sends data to the database
    """

    # Pull cities table from database
    cities_table = pd.read_sql('cities',
                               con=connection_string)

    # Create empty list for storing data
    populations_data = []

    # Loop through cities in cities table

    for city in cities_table['city']:
        url = f"https://www.wikipedia.org/wiki/{city}"
        headers = {'User-Agent': 'Chrome/134.0.0.0'}

        response = requests.get(url, headers=headers)
        city_soup = BeautifulSoup(response.content, 'html.parser')

        # Extract the population data
        city_population = city_soup.find(string="Population").find_next("td").get_text()
        city_population_clean = int(city_population.replace(",", ""))

        timestamp = (dt.datetime
                     .now(timezone('Europe/Berlin'))
                     .strftime("%Y-%m-%d %H:%M:%S")
                    )

        populations_data.append({"city": city,
                                "population": city_population_clean,
                                "timestamp_population": timestamp})


    # Combine all rows into data frame
    populations_df = pd.DataFrame(populations_data)

    # Get foreign key ids from cities table
    city_populations_df = (
        populations_df
        .merge(cities_table,
               on='city',
               how='inner')
        [['city_id', 'population', 'timestamp_population']]
    )
    # Send data to database
    city_populations_df.to_sql('population',
                               if_exists='append',
                               con=connection_string,
                               index=False)

## Extract weather forecasts

In [5]:
def update_weather_table(connection_string: str) -> None:
    """
    Description: Requests 5 day weather forecasts for all cities in the connected database
    Parameters:
        connection_string | str: string for SQLAlchemy connections
    Returns:
        None, just updates database
    """

    # Pull city data from database to get corresponding forecasts
    cities_df = pd.read_sql('cities', con=connection_string)

    forecasts = []

    # Loop through all the cities in the cities table pulled from the database
    for _, row in cities_df.iterrows():
        # Gather coordinates for API request
        lat = row['latitude']
        long = row['longitude']

        # Make API request
        params = {
            'lat': lat,
            'lon': long,
            'appid': os.getenv("openWeatherApi"),
            'units': "metric",
        }
        response = requests.get(url='https://api.openweathermap.org/data/2.5/forecast?',
                                params=params)
        weather_data = response.json()

        retrieval_time = (dt.datetime
                          .now(timezone('Europe/Berlin'))
                          .strftime("%Y-%m-%d %H:%M:%S")
                          )

        # Loop through all the forecasts
        for forecast in weather_data['list']:
            # Locate desired information
            forecast_dict = {
                'city_id': row['city_id'],
                'forecast_time': forecast.get("dt_txt"),
                'temperature': forecast["main"].get("temp"),
                "forecast": forecast["weather"][0].get("main"),
                'rain_in_last_3h': forecast.get('rain', {'3h':0})['3h'],
                "wind_speed": forecast["wind"].get("speed"),
                "data_retrieved_at": retrieval_time,
            }
            forecasts.append(forecast_dict)

    weather_df = pd.DataFrame(forecasts)
    weather_df.to_sql('weather',
                        if_exists='append',
                        con=connection_string,
                        index=False)

## Get airports nearby

In [10]:
def request_airports_data(
        cities: list[str],
        connection_string: str,
        ) -> None:
    """
    Description: requests airport data from API for newly added cities,
                 then sends data to the airports table and a bridge table
                 to the connected database
    Parameters:
        cities_list | list : list of city names
        connection_string | str: string for SQLAlchemy connections
    Returns:
        None, just sends data to the database
    """

    # Pull cities table from database
    cities_df = pd.read_sql('cities', con=connection_string)

    # Create lists to store data
    airports_data = []
    airports_cities_data = []
    # Loop through new cities
    for city in cities:

        row = cities_df.loc[cities_df['city'] == city].iloc[0]

        # Construct request
        url = "https://aerodatabox.p.rapidapi.com/airports/search/location"
        params = {"withFlightInfoOnly":"true",
                    "lat":row['latitude'],
                    'lon':row['longitude'],
                    'radiusKm':"50",
                    'limit':10}
        headers = {
            "X-RapidAPI-Key": os.getenv("x_rapidapi_key"),
            "X-RapidAPI-Host": "aerodatabox.p.rapidapi.com",
        }
        # Request from API
        sleep(5)
        response = requests.get(url, headers=headers, params=params)
        response.raise_for_status()
        airports_json = response.json()

        # Loop through airports in response
        for airport in airports_json['items']:
            # Gather data for database
            airport_data = {
                'icao': airport['icao'],
                'iata': airport['iata'],
                'airport_name': airport['name'],
                'latitude': airport['location']['lat'],
                'longitude': airport['location']['lon']
            }
            # Add to list of airport data
            airports_data.append(airport_data)

            # Store airport-city connection for bridge table
            airport_city_data = {
                'icao': airport['icao'],
                'city_id': row['city_id'],
            }
            # Add to list of airport-city connections
            airports_cities_data.append(airport_city_data)

    # Convert to data frames
    airports_df = pd.DataFrame(airports_data)
    # Remove any duplicate airports
    airports_df = airports_df.drop_duplicates()
    cities_airports_df = pd.DataFrame(airports_cities_data)

    # Send data to database
    airports_df.to_sql(
        'airports',
        if_exists='append',
        con=connection_string,
        index=False)
    cities_airports_df.to_sql(
        'cities_airports',
        if_exists='append',
        con=connection_string,
        index=False)

## Get flight data for next day

In [11]:
def request_flights_data(connection_string: str) -> None:

    # Pull airports from database to get corresponding flights
    airports_df = pd.read_sql('airports', con=connection_string)

    flights_data = []

    # Create time windows for 12hr searches
    # Get datetime of tomorrow
    tomorrow = dt.datetime.now(timezone('Europe/Berlin')) + dt.timedelta(days=1)
    # Grab just the date
    tomorrow_date = tomorrow.strftime('%Y-%m-%d')
    # Manually add the rest to get the start and end of the day
    morning_start = f'{tomorrow_date}T00:00'
    morning_end = f'{tomorrow_date}T11:59'
    afternoon_start = f'{tomorrow_date}T12:00'
    afternoon_end = f'{tomorrow_date}T23:59'
    day_parts = [(morning_start, morning_end), (afternoon_start, afternoon_end)]

    # Loop through airports
    for _, row in airports_df.iterrows():
        # Loop for both halves of day
        for time_start, time_end in day_parts:

            # Construct request
            base_url = "https://aerodatabox.p.rapidapi.com/flights/airports"
            path_params = f"/icao/{row['icao']}/{time_start}/{time_end}"
            full_url = base_url + path_params
            params = {
                'withLeg':True,
                'direction':'Arrival',
                'withCancelled':False,
                'withCodeshared':False,
                'withCargo':False,
                'withPrivate':False,
                'withLocation':False
            }
            headers = {
                "X-RapidAPI-Key": os.getenv("x_rapidapi_key"),
                "x-rapidapi-host": "aerodatabox.p.rapidapi.com",
            }
            # Request data from API
            sleep(5)
            response = requests.get(full_url, headers=headers, params=params)
            if(response.status_code == 200):
                flights_json = response.json()

                # Loop through flights
                for flight in flights_json['arrivals']:

                    # Gather data
                    scheduled_arrival = pd.to_datetime(flight
                                                       ['arrival']
                                                       ['scheduledTime']
                                                       ['local']).replace(tzinfo=None)
                    # If there is no revised time, set it to the scheduled time
                    updated_arrival = pd.to_datetime(
                        flight['arrival']
                        .get('revisedTime', flight['arrival']['scheduledTime'])
                        ['local'],
                    ).replace(tzinfo=None)
                    # Arrange data in dictionary
                    flight_data = {
                        'flight_number': flight['number'],
                        'icao': row['icao'],
                        'scheduled_arrival_time': scheduled_arrival,
                        'updated_arrival_time': updated_arrival,
                        'departure_airport': flight['departure']['airport']['name'],
                    }
                    flights_data.append(flight_data)
            else:
                print(response.status_code, "code for", row['icao'])

    flights_df = pd.DataFrame(flights_data)
    # Send data to database
    try:
        flights_df.to_sql('flights',
                        if_exists='append',
                        con=connection_string,
                        index=False)
    except:
        print("Flights could not be send to SQL database.\n"
              "Maybe you run this function already today.\n"
              "Please come back tomorrow.")

# 3. Call functions

## Generate connection string used by all other functions

In [12]:
connection_string = connect_to_gans_local()

## Scrape city information for (new) cities and extract immediately airport information too

Note: `request_airports_data` gets the cities as input instead of extracting them from SQL to run only for newly added cities.

In [13]:
cities = ["Berlin", "Munich", "Hamburg"]
scrape_and_send_city_data(cities, connection_string)
request_airports_data(cities, connection_string)

## Scrape population data (Can run once a year for example)

In [14]:
scrape_and_send_population_data(connection_string)

## Get latest weather forecast (Should run at least once a day)

In [15]:
update_weather_table(connection_string)

## Get flights for next day (Should run once a day)

In [16]:
request_flights_data(connection_string)

204 code for EDDT
204 code for EDDT
